# Satellite Land Cover Classification

**Note:** The included 700-row dataset is synthetic educational data, not real satellite observations.

## 1. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import joblib

## 2. Load and inspect dataset

In [ ]:
df = pd.read_csv("dataset/Satellite_Land_Cover_700_Sample.csv")
display(df.head())
print("Shape:", df.shape)
print(df["Land_Cover"].value_counts())

## 3. Separate features and target

In [ ]:
features = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2", "NDVI"]
X = df[features]
y = df["Land_Cover"]

## 4. Split training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 5. Train and compare algorithms

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", probability=True, random_state=42))]),
    "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=5))])
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({"Algorithm": name, "Accuracy": accuracy_score(y_test, pred)})
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df)

## 6. Evaluate and save the best model

In [ ]:
best_name = results_df.iloc[0]["Algorithm"]
best_model = models[best_name]
y_pred = best_model.predict(X_test)
print("Best model:", best_name)
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation=45)
plt.tight_layout()
plt.show()
joblib.dump(best_model, "models/land_cover_model.joblib")
results_df.to_csv("results/model_comparison.csv", index=False)

## 7. Important limitation

These results measure performance only on the synthetic demonstration dataset. They do not establish accuracy on real satellite imagery.